In [ ]:
from system.converter_HTML_to_PDF import gera_html
from function_bd import integridade_bd, save_json, leitura_de_contas
from function_tela_atividades import progresso_lista
from class_papiron.class_dados_aluno import Disciplina
from selenium import webdriver
# from seleniumwire import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup as BS
from datetime import datetime
import time
import os
import shutil
import json
import fnmatch

from function import  login, iniciar_sesssao_chrome, painel_disciplinas, ult_div, atualizar_soup, ignorar_aviso

user = "21187764-5"
senha = "Celio.999522446"

user = "222708735"
senha = "Lu4258"
driver = iniciar_sesssao_chrome(True)

login(driver,user,senha)

# d = "file:///C:/Users/blitz/Desktop/Papiron/202351/GRADUA%C3%87%C3%83O%20EM%20ADMINISTRA%C3%87%C3%83O/ADMINISTRA%C3%87%C3%83O%20DE%20MARKETING/ADM%20-%20MAPA%20-%20ADMINISTRA%C3%87%C3%83O%20DE%20MARKETING%20-%20202351.html"
# driver.get(d)
# time.sleep(1)
# driver.execute_script('window.print();')

ModuleNotFoundError: No module named 'converter_HTML_to_PDF'

In [4]:
d = "file:///C:/Users/blitz/Desktop/Papiron/202351/GRADUA%C3%87%C3%83O%20EM%20ADMINISTRA%C3%87%C3%83O/ADMINISTRA%C3%87%C3%83O%20DE%20MARKETING/ADM%20-%20MAPA%20-%20ADMINISTRA%C3%87%C3%83O%20DE%20MARKETING%20-%20202351.html"
driver.get(d)
time.sleep(1)
driver.execute_script('window.print();')

In [ ]:
btn_material = WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.PARTIAL_LINK_TEXT,"Material da Disciplina"))) 
driver.execute_script("arguments[0].click();",btn_material)
ult_div(driver)

In [ ]:
url_painel_disciplina = driver.current_url

# Clicar no Botão MATERIAL DA DISCIPLINA
btn_material = WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.PARTIAL_LINK_TEXT,"Material da Disciplina"))) 
driver.execute_script("arguments[0].click();",btn_material)
ult_div(driver)

soup = atualizar_soup(driver)
links_downloads = soup.find_all('span',{'alt':'fazer download'})
count_download = len(links_downloads)
name_download = soup.find_all('span',{'class':'bold all-caps font-montserrat fileNameCrop ng-binding'})
curso = "ADM"
mod = "202351"
disciplina = "DISC_TESTE"
path_download = os.path.join(os.path.expanduser("~"), "Desktop\\Papiron\\Downloads\\Temp")
path_papiron = os.path.join(os.path.expanduser("~"), "Desktop\\Papiron")
path_destino = path_papiron+'\\'+mod+'\\'+curso+'\\'+disciplina
# Verifica pasta do mod
if not os.path.isdir(path_papiron+'\\'+mod):
    os.mkdir(path_papiron+'\\'+mod)

# Verifica pasta do CURSO
if not os.path.isdir(path_papiron+'\\'+mod+'\\'+curso):
    os.mkdir(path_papiron+'\\'+mod+'\\'+curso)

# Verifica pasta da DISCIPLINA
if not os.path.isdir(path_papiron+'\\'+mod+'\\'+curso+'\\'+disciplina):
    os.mkdir(path_papiron+'\\'+mod+'\\'+curso+'\\'+disciplina)


In [ ]:
from function import ignorar_aviso

# Remove os arquivos temporários que falharam
files = os.listdir(path_download)

for file in files:
    os.remove(file)

for i in range(count_download):
    try:
        name = name_download[i].text
        filename = path_papiron+'\\'+mod+'\\'+curso+'\\'+disciplina+'\\'+name_download[i].text
        soup.find_all('span',{'class':'bold all-caps font-montserrat fileNameCrop ng-binding'})
        css_download = '#content > div > ui-view > ui-view > ui-view > div > div > div > div.container-fluid > div > div.col-lg-8 > div > ng-repeat:nth-child('+str(i+1)+') > div > div > div.panel-footer.no-border.no-padding > div > div > div > div > a > span'
        btn_download = WebDriverWait(driver, 1).until(EC.element_to_be_clickable((By.CSS_SELECTOR,css_download)))
        driver.execute_script("arguments[0].click();",btn_download)
        print("Download: ",name)
        time.sleep(3)
        driver.refresh()
        ignorar_aviso(driver,True)
        ult_div(driver)

        # se a lista de arquivos incompletos estiver vazia, então todos os downloads foram finalizados
        d =True
        cont = 100
        while d and cont:
            # lista todos os arquivos da pasta de destino
            files = os.listdir(path_download)

            # filtra apenas os arquivos com as extensões .crdownload ou .part
            incomplete_files = fnmatch.filter(files, '*.crdownload') + fnmatch.filter(files, '*.part')

            if not incomplete_files:

                print('Todos os downloads foram finalizados.')
                
                # Renomeia para o Nome Texto
                files = os.listdir(path_download)
                extensao = files[0][files[0].rfind("."):]
                os.rename(path_download+'\\'+files[0],path_download+'\\'+name+extensao)

                # Verifica se não arquivo na pasta de destino
                if os.path.isfile(path_destino+'\\'+name+extensao):
                    os.remove(path_destino+'\\'+name+extensao)
                
                # Move para o arquivo de destino
                shutil.move(path_download+'\\'+name+extensao,path_destino+'\\'+name+extensao)
                d = False

            else:
                print('Existem downloads em progresso.')
                time.sleep(3)
                cont-=1
                
        # driver.voltar uma
    except TimeoutException:
        print("ERRO")
        btn_template = WebDriverWait(driver, 1).until(EC.element_to_be_clickable((By.PARTIAL_LINK_TEXT,"padrao")))

In [ ]:
import io

from system.converter_HTML_to_PDF import html_to_pdf

# Retirar estes trechos
a = "<div class=\"alert alert-danger ng-hide\" ng-show=\"vm.tempoMax &gt; 0 &amp;&amp; !vm.questionarioDisabled &amp;&amp; !vm.questionario.finalizado\" style=\"text-align:center;margin:0 auto 15px;width:50%\"> <span>Restam <b class=\"ng-binding\"> </b> para o fim da Atividade Avaliativa</span> </div>"
b = "<div class=\"alert alert-danger ng-hide\" ng-show=\"vm.tempoMax &gt; 0 &amp;&amp; vm.questionario.situacao.descricao !== 'ABERTO' &amp;&amp; vm.questionarioDisabled\" style=\"text-align:center;margin:0 auto 15px;width:50%\"> <span><b>Data final atingida! A Atividade Avaliativa está indisponível.</b></span> </div>" 
c = "<div class=\"alert alert-danger ng-hide\" ng-show=\"vm.tempoMax &gt; 0 &amp;&amp; vm.questionario.situacao.descricao === 'ABERTO' &amp;&amp; vm.isTimedout\" style=\"text-align:center;margin:0 auto 15px;width:50%\"> <span><b>Tempo esgotado ! A Atividade Avaliativa está indisponível.</b></span> </div> "
d = "<span class=\"label label-danger all-caps pull-right m-r-20 ng-hide\" ng-hide=\"!vm.questaoAtual.anulada\"> anulada </span> </div>"

soup = atualizar_soup(driver)
cab = str(soup.find('div',{'class':'panel panel-default ng-scope'}))
body = soup.find('div',{"id":"cabecalhoQuestao"})
body = str(body).replace(a,"").replace(b,"").replace(c,"").replace(d,"")

alternativas = soup.find_all('div',{"class":"inline-block m-t-10 ng-binding"})
for alternativa in alternativas:
    print(alternativa.text)

html_inic = "<!DOCTYPE html><html><head><link rel=\"stylesheet\" type=\"text/css\" href=\"unicesumar.css\"><meta charset=\"UTF-8\" /><title>PAPIRON</title></head><body>"
html_fim = "</body></html>"
html = html_inic+"<br><br>"+'<div class=\"container\" style=\"border: 0px solid;\">'+"<br><br>"+cab+"<br><br>"+body+'</div>'+html_fim

path_file_html = os.path.abspath(os.getcwd())+'\\BD\\atividades\\teste.html'
path_file_pdf = os.path.abspath(os.getcwd())+'\\BD\\atividades\\teste.pdf'

with io.open(path_file_html, 'w', encoding='utf8') as arquivo:
    arquivo.write(html)

html_to_pdf(path_file_html, path_file_pdf)

In [ ]:
driver.refresh()


In [ ]:
from system.converter_HTML_to_PDF import html_to_pdf


soup = atualizar_soup(driver)

alternativas = []
enunciados = []
enunciado =  soup.find('div',{"class":"enunciado ng-binding"})

alternativas.append(soup.find_all('div',{"class":"inline-block m-t-10 ng-binding"}))
enunciados.append(enunciado)


if not enunciados:
    cab = str(soup.find('div',{'class':'panel panel-default ng-scope'}))
    body = str(soup.find('div',{"id":"cabecalhoQuestao"})+"<br><br>")
else:
    body = ""
    for i, enunciado in enumerate(enunciados):
        body = body+"<br><br><b>QUESTÃO "+str(i+1)+"</b><br>"+ str(enunciado)
        for j, alternativa in enumerate(alternativas[i]):
            body = body+"<br><input type=\"checkbox\"> <b>ALTERNATIVA "+str(j+1)+"</b>"+str(alternativa)+"<br>"


body = str(body).replace(a,"").replace(b,"").replace(c,"").replace(d,"")

html_inic = "<!DOCTYPE html><html><head><link rel=\"stylesheet\" type=\"text/css\" href=\"../../unicesumar.css\"><meta charset=\"UTF-8\" /><title>PAPIRON</title></head><body>"
html_fim = "</body></html>"
html = html_inic+"<br><br>"+'<div class=\"container\" style=\"border: 0px solid;\">'+"<br>"+cab+body+'</div>'+html_fim

# path_file_html = path_file+'.html'
# path_file_pdf = path_file+'.pdf'
path_file_html = os.path.abspath(os.getcwd())+'\\BD\\atividades\\teste.html'
path_file_pdf = os.path.abspath(os.getcwd())+'\\BD\\atividades\\teste.pdf'

with io.open(path_file_html, 'w', encoding='utf8') as arquivo:
    arquivo.write(html)

html_to_pdf(path_file_html, path_file_pdf)

In [ ]:
soup = atualizar_soup(driver)
# Clicar no Botão MATERIAL DA DISCIPLINA
btn_material = WebDriverWait(driver, 30).until(EC.element_to_be_clickable((By.PARTIAL_LINK_TEXT,"Material da Disciplina"))) 
driver.execute_script("arguments[0].click();",btn_material)

In [ ]:
driver.execute_script('window.print();')